[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

© Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es

&copy; José Luis Chiquete Valdivieso. 2017-2026.

# Cuantización y ventana de contexto: memoria y velocidad.

**py521 · IA Generativa en Entornos Locales** · Pythonista® · Instructor: José Luis Chiquete · Demo del Tema 4: *Ecosistema de ejecución local*

En este *notebook* se cargará el mismo modelo en dos cuantizaciones (por omisión, 4 y 8 bits) y con dos ventanas de contexto. Se compararán la memoria ocupada, el tiempo al primer token y los tokens por segundo.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* Las dos versiones se eligen con `LLM_MODEL_A` y `LLM_MODEL_B` (por omisión, `qwen3:1.7b-q4_K_M` y `qwen3:1.7b-q8_0`; las dos se descargan con `ollama pull`, ver `00_preliminares_entorno_y_secretos.ipynb`); las ventanas, con `LLM_CONTEXT_A` y `LLM_CONTEXT_B` (por omisión, 4096 y 16384 tokens).
* La sección 1 funciona con cualquier motor compatible con *OpenAI*; la memoria y la sección 2 usan la API propia de *Ollama*.
* Memoria necesaria con los valores por omisión: unos 4 GB libres de RAM, VRAM o memoria unificada.

### Configuración común.

Esta celda es **idéntica en todos los *notebooks***: lee la configuración de variables de entorno (o del archivo `.env`, si existe) y usa valores por omisión cuando no están definidas. Muestra todas las variables, pero **en esta demo importan: `LLM_BASE_URL`, `LLM_MODEL_A`, `LLM_MODEL_B`, `LLM_CONTEXT_A`, `LLM_CONTEXT_B` y, con *Ollama*, `OLLAMA_URL`**. Las claves solo se muestran como «definida / no definida». La tabla completa de variables está en el `README.md` y en `00_preliminares_entorno_y_secretos.ipynb`.

In [ ]:
# Configuración común de py521 (celda idéntica en todos los notebooks).
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)  # las variables del sistema tienen prioridad sobre .env

def _entorno(nombre, por_omision=None):
    valor = os.environ.get(nombre, "").strip()
    return valor or por_omision

# Servicio compatible con OpenAI (Ollama, LM Studio, llama.cpp server, vLLM...).
LLM_BASE_URL = _entorno("LLM_BASE_URL", _entorno("LLM_DEFAULT_BASE_URL", "http://localhost:11434/v1")).rstrip("/")  # el contenedor fija LLM_DEFAULT_BASE_URL
LLM_API_KEY = _entorno("LLM_API_KEY")              # secreto: sin valor por omisión
LLM_MODEL = _entorno("LLM_MODEL", "qwen3:1.7b")
LLM_EMBED_MODEL = _entorno("LLM_EMBED_MODEL", "qwen3-embedding:0.6b")
LLM_MODEL_A = _entorno("LLM_MODEL_A", "qwen3:1.7b-q4_K_M")
LLM_MODEL_B = _entorno("LLM_MODEL_B", "qwen3:1.7b-q8_0")
LLM_CONTEXT_A = int(_entorno("LLM_CONTEXT_A", "4096"))
LLM_CONTEXT_B = int(_entorno("LLM_CONTEXT_B", "16384"))
RAG_DOCS_DIR = Path(_entorno("RAG_DOCS_DIR", "data"))
LLM_PROXY_KEY = _entorno("LLM_PROXY_KEY")          # secreto de la demo 05: sin valor por omisión
LLM_PROXY_HOST = _entorno("LLM_PROXY_HOST", "127.0.0.1")
LLM_PROXY_PORT = int(_entorno("LLM_PROXY_PORT", "8800"))

# API nativa de Ollama (solo para las funciones marcadas «específico de Ollama»).
# Se usa OLLAMA_HOST, como el cliente oficial de Ollama; si no existe, se deriva de LLM_BASE_URL.
def _url_ollama(valor):
    valor = (valor or LLM_BASE_URL.removesuffix("/v1")).strip().rstrip("/")
    if "://" not in valor:
        anfitrion = valor.split("/")[0]
        valor = "http://" + (valor if ":" in anfitrion.split("]")[-1] else valor.replace(anfitrion, anfitrion + ":11434", 1))
    return valor.replace("://0.0.0.0", "://localhost")  # 0.0.0.0 es una dirección de escucha, no de destino

OLLAMA_URL = _url_ollama(_entorno("OLLAMA_HOST"))

for nombre in ["LLM_BASE_URL", "LLM_MODEL", "LLM_EMBED_MODEL", "LLM_MODEL_A", "LLM_MODEL_B",
               "LLM_CONTEXT_A", "LLM_CONTEXT_B", "RAG_DOCS_DIR", "LLM_PROXY_HOST", "LLM_PROXY_PORT", "OLLAMA_URL"]:
    print(f"{nombre:<16} {globals()[nombre]}")
for nombre in ["LLM_API_KEY", "LLM_PROXY_KEY", "HF_TOKEN"]:
    print(f"{nombre:<16} {'definida' if _entorno(nombre) else 'no definida'}")

### Preparación del entorno.

* Las siguientes celdas crearán el cliente de la API compatible con *OpenAI*, comprobarán que los modelos están disponibles y definirán las funciones de la demo; no hace falta leerlas.

In [ ]:
from openai import OpenAI

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY or "sin-clave")  # Ollama y LM Studio sin autenticación ignoran la clave

def comprobar_servicio(modelos):
    """Comprueba que el servicio responde (GET /models) y que ofrece los modelos indicados."""
    try:
        disponibles = {m.id for m in cliente.models.list().data}
    except Exception as error:
        raise RuntimeError(f"No responde el servicio en {LLM_BASE_URL} ({type(error).__name__}). "
                           "Abre Ollama o el motor que uses, o revisa LLM_BASE_URL.") from None
    faltan = [m for m in modelos if m not in disponibles and f"{m}:latest" not in disponibles]
    if faltan:
        raise RuntimeError(f"El servicio no ofrece: {', '.join(faltan)}. En Ollama: "
                           + " · ".join(f"ollama pull {m}" for m in faltan))
    print(f"Servicio en {LLM_BASE_URL}: responde. Modelos: {', '.join(modelos)}.")

comprobar_servicio([LLM_MODEL_A, LLM_MODEL_B])

In [ ]:
import time

def preguntar(prompt, modelo=None, temperatura=None, razonamiento=None, max_tokens=None, sistema=None):
    """Envía una petición a /chat/completions en modo streaming y mide los tiempos.

    razonamiento: None (valor del modelo), "none" (desactivado) o "low" / "medium" / "high".
    Se envía como `reasoning_effort` (Ollama, vLLM, LM Studio) y como
    `chat_template_kwargs.enable_thinking` (llama.cpp server, vLLM con Qwen3); cada motor ignora lo que no usa.
    """
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": prompt}]
    extra, argumentos = {}, {}
    if temperatura is not None:
        argumentos["temperature"] = temperatura
    if max_tokens is not None:
        argumentos["max_tokens"] = max_tokens
    if razonamiento is not None:
        argumentos["reasoning_effort"] = razonamiento
        extra["chat_template_kwargs"] = {"enable_thinking": razonamiento != "none"}
    inicio = time.perf_counter()
    primer_token = primer_texto = None
    texto, pensado, uso, fin = [], [], None, None
    flujo = cliente.chat.completions.create(model=modelo or LLM_MODEL, messages=mensajes, stream=True,
                                            stream_options={"include_usage": True}, extra_body=extra or None,
                                            **argumentos)
    for parte in flujo:
        if parte.usage:
            uso = parte.usage
        if not parte.choices:
            continue
        delta = parte.choices[0].delta
        extras = delta.model_extra or {}
        trozo_razonamiento = extras.get("reasoning") or extras.get("reasoning_content")  # el nombre varía por motor
        ahora = time.perf_counter() - inicio
        if primer_token is None and (delta.content or trozo_razonamiento):
            primer_token = ahora
        if primer_texto is None and delta.content:
            primer_texto = ahora
        pensado.append(trozo_razonamiento or "")
        texto.append(delta.content or "")
        fin = parte.choices[0].finish_reason or fin
    total = time.perf_counter() - inicio
    tokens = uso.completion_tokens if uso else 0
    generacion = total - (primer_token or total)
    return {
        "respuesta": "".join(texto).strip(),
        "razonamiento": "".join(pensado).strip(),
        "primer_token_s": primer_token,
        "primer_texto_s": primer_texto,
        "total_s": total,
        "tokens": tokens,
        "tokens_s": tokens / generacion if generacion > 0 else 0.0,
        "fin": fin,
    }

In [ ]:
# ── 🦙 Específico de Ollama ────────────────────────────────────────────────
# Estas funciones usan la API nativa de Ollama (OLLAMA_URL). Con otro motor
# devuelven None y muestran un aviso; el resto del notebook sigue funcionando.
import httpx

def ollama_version():
    """Versión de Ollama, o None si OLLAMA_URL no es un servicio Ollama."""
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=5).json()["version"]
    except Exception:
        return None

ES_OLLAMA = ollama_version() is not None
print(f"Ollama en {OLLAMA_URL}:", f"sí (versión {ollama_version()})" if ES_OLLAMA
      else "no detectado; las funciones específicas de Ollama se omitirán")

_avisos_mostrados = set()

def aviso_no_ollama(que, equivalente):
    """Muestra una sola vez por notebook que una medición es específica de Ollama y su equivalente."""
    if que not in _avisos_mostrados:
        _avisos_mostrados.add(que)
        print(f"ℹ️ {que} solo se mide con Ollama. Equivalente en otros motores: {equivalente}")

def ollama_memoria(modelo):
    """🦙 Memoria del modelo cargado según /api/ps (total, parte en VRAM y ventana de contexto)."""
    if not ES_OLLAMA:
        aviso_no_ollama("La memoria ocupada", "LM Studio: indicador de memoria de la aplicación; llama.cpp server: "
                        "registro al cargar el modelo; vLLM: `nvidia-smi` o el registro de arranque; "
                        "en todos: el monitor del sistema.")
        return None
    for m in httpx.get(f"{OLLAMA_URL}/api/ps", timeout=10).json().get("models", []):
        if m["name"] == modelo or m["model"] == modelo:
            return {"total_gb": m["size"] / 1e9, "vram_gb": m.get("size_vram", 0) / 1e9,
                    "contexto": m.get("context_length")}
    return None

def ollama_descargar(modelo):
    """🦙 Saca el modelo de la memoria (keep_alive = 0) para medir la siguiente carga desde cero."""
    if ES_OLLAMA:
        httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": modelo, "keep_alive": 0}, timeout=60)


def ollama_medir_con_contexto(prompt, modelo, num_ctx, max_tokens=128):
    """🦙 Carga el modelo con una ventana de contexto (num_ctx) y mide memoria y velocidad.

    La API compatible con OpenAI no permite elegir la ventana; por eso se usa /api/chat de Ollama.
    """
    if not ES_OLLAMA:
        aviso_no_ollama("Cambiar la ventana por petición", "la ventana se fija al cargar el modelo: "
                        "LM Studio (longitud de contexto en las opciones de carga del modelo), "
                        "llama.cpp server (`-c`), vLLM (`--max-model-len`). Carga dos veces y usa LLM_MODEL_A/B.")
        return None
    cuerpo = {"model": modelo, "messages": [{"role": "user", "content": prompt}], "stream": False,
              "think": False, "options": {"num_ctx": num_ctx, "num_predict": max_tokens, "temperature": 0}}
    final = httpx.post(f"{OLLAMA_URL}/api/chat", json=cuerpo, timeout=900).json()
    memoria = ollama_memoria(modelo) or {}
    generacion = final.get("eval_duration", 0) / 1e9
    return {"memoria_gb": memoria.get("total_gb"), "contexto": memoria.get("contexto"),
            "primer_token_s": (final.get("load_duration", 0) + final.get("prompt_eval_duration", 0)) / 1e9,
            "tokens_s": final.get("eval_count", 0) / generacion if generacion else 0.0}

In [ ]:
import matplotlib.pyplot as plt

PROMPT = "Describe en cinco frases cómo se prepara una taza de té."
REPETICIONES = 3

def graficar(resultados, titulo, etiqueta_primer_token="Primer token (s)"):
    """Tres barras por configuración: memoria, tiempo al primer token y tokens por segundo."""
    nombres = list(resultados)
    metricas = [("Memoria (GB)", "memoria_gb"), (etiqueta_primer_token, "primer_token_s"), ("Tokens por segundo", "tokens_s")]
    fig, ejes = plt.subplots(1, 3, figsize=(12, 3.6))
    for eje, (etiqueta, clave) in zip(ejes, metricas):
        valores = [resultados[n].get(clave) or 0 for n in nombres]
        eje.bar(nombres, valores, color=["#4C72B0", "#DD8452"])
        eje.set_title(etiqueta)
        eje.tick_params(axis="x", labelsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

## 1. Dos cuantizaciones del mismo modelo.

* **Cuantización:** guardar los pesos con menos bits. Q4 usa unos 4 bits por peso y Q8 unos 8: Q4 ocupa menos memoria; Q8 se acerca más al modelo original. La velocidad depende del equipo: la gráfica muestra cuál es más rápida en este equipo.
* Para cada versión se descarga primero el modelo de la memoria, se hace una petición de carga y luego `REPETICIONES` peticiones medidas. Cada petición cambia la primera línea para que cada medición empiece desde cero (el motor guarda lo que ya procesó y respondería al instante).

In [ ]:
cuantizaciones = {}
for modelo in [LLM_MODEL_A, LLM_MODEL_B]:
    ollama_descargar(modelo)
    carga = preguntar(f"Carga.\n{PROMPT}", modelo=modelo, razonamiento="none", temperatura=0.0, max_tokens=16)
    medidas = [preguntar(f"Intento {i}.\n{PROMPT}", modelo=modelo, razonamiento="none", temperatura=0.0, max_tokens=128)
               for i in range(1, REPETICIONES + 1)]
    memoria = ollama_memoria(modelo) or {}
    cuantizaciones[modelo] = {
        "memoria_gb": memoria.get("total_gb"),
        "carga_s": carga["total_s"],
        "primer_token_s": sum(m["primer_token_s"] for m in medidas) / REPETICIONES,
        "tokens_s": sum(m["tokens_s"] for m in medidas) / REPETICIONES,
    }
    ollama_descargar(modelo)
    r = cuantizaciones[modelo]
    memoria_txt = f"{r['memoria_gb']:.2f} GB" if r["memoria_gb"] else "no disponible"
    print(f"{modelo:<22} memoria {memoria_txt} · carga {r['carga_s']:.1f} s · "
          f"primer token {r['primer_token_s']:.2f} s · {r['tokens_s']:.1f} tokens/s")

In [ ]:
graficar(cuantizaciones, "Misma petición, dos cuantizaciones")

* La memoria ocupada incluye los pesos y la caché de contexto (caché KV): la memoria donde el motor guarda lo ya procesado de la ventana de contexto.
* La diferencia de calidad entre Q4 y Q8 no se mide aquí: depende del modelo y de la tarea.

## 2. Dos ventanas de contexto del mismo modelo.

* La **ventana de contexto** es el máximo de tokens que el modelo atiende a la vez. Reservar una ventana mayor aumenta la memoria de la caché de contexto (caché KV) aunque el *prompt* sea corto.
* Se usa `LLM_MODEL_A` con `LLM_CONTEXT_A` y `LLM_CONTEXT_B` tokens.
* Aquí el primer token se calcula con los tiempos que informa *Ollama* (carga más lectura del *prompt*), no con el reloj del *notebook* como en la sección 1.

In [ ]:
ventanas = {}
for num_ctx in [LLM_CONTEXT_A, LLM_CONTEXT_B]:
    ollama_descargar(LLM_MODEL_A)
    if ollama_medir_con_contexto(f"Carga.\n{PROMPT}", LLM_MODEL_A, num_ctx, max_tokens=1) is None:
        break  # el motor no es Ollama: ya se mostró el aviso con el equivalente
    medidas = [ollama_medir_con_contexto(f"Intento {i}.\n{PROMPT}", LLM_MODEL_A, num_ctx)
               for i in range(1, REPETICIONES + 1)]
    ventanas[f"{num_ctx} tokens"] = {
        "memoria_gb": medidas[-1]["memoria_gb"],
        "primer_token_s": sum(m["primer_token_s"] for m in medidas) / REPETICIONES,
        "tokens_s": sum(m["tokens_s"] for m in medidas) / REPETICIONES,
    }
    r = ventanas[f"{num_ctx} tokens"]
    print(f"Ventana {num_ctx:>6} (cargada: {medidas[-1]['contexto']}): memoria {r['memoria_gb']:.2f} GB · "
          f"primer token {r['primer_token_s']:.2f} s · {r['tokens_s']:.1f} tokens/s")
ollama_descargar(LLM_MODEL_A)

In [ ]:
if ventanas:
    graficar(ventanas, f"{LLM_MODEL_A} con dos ventanas de contexto", "Primer token (s, según Ollama)")

* Con un *prompt* corto, la velocidad apenas cambia; la memoria sí, porque la caché de contexto se reserva para toda la ventana.
* Si la memoria está al límite: versión con menos bits o ventana más corta.

---
## ✅ Cierre.

* Menos bits por peso: menos memoria; la velocidad depende del equipo y se mide en cada caso.
* Una ventana de contexto mayor reserva más memoria aunque no se use.
* Las cifras dependen del equipo; lo que se compara es la diferencia entre configuraciones.

**Siguiente *notebook*:** `04_rag_local_con_fuente_citada.ipynb`.

<p style="text-align: center"><a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>